# Análisis de Caso: Reducción Dimensional
**Empresa:** VisionData | **Rol:** Especialista en Ciencia de Datos

Este notebook aplica PCA y t-SNE sobre un dataset sintético de encuestas masivas con 60 variables por cliente, representando el contexto del caso de VisionData.

## 1. Importación de librerías

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

print('Librerías importadas correctamente.')

## 2. Carga del dataset
Generamos un dataset sintético que simula encuestas masivas de clientes con:
- 500 registros (clientes)
- 60 variables: 20 demográficas, 20 de hábitos de consumo, 20 de preferencias digitales
- 4 segmentos naturales de clientes (etiqueta de referencia)

In [ ]:
np.random.seed(42)
n_features = 60
segment_names = ['Jovenes_digitales', 'Adultos_conservadores', 'Seniors_offline', 'Profesionales_tech']
labels_pretty = ['Jóvenes digitales', 'Adultos conservadores', 'Seniors offline', 'Profesionales tech']

# Cada segmento tiene un centroide diferente en el espacio de 60 dimensiones
offsets = [
    [2]*20 + [-1]*20 + [0]*20,   # Jóvenes digitales: alto en demo, bajo en consumo
    [-2]*20 + [1]*20 + [0]*20,   # Adultos conservadores: bajo en demo
    [0]*20 + [-2]*20 + [2]*20,   # Seniors offline: alto en preferencias digitales
    [1]*20 + [2]*20 + [-2]*20    # Profesionales tech: alto consumo, bajo digital
]

X_parts, y_parts = [], []
for name, off in zip(segment_names, offsets):
    data = np.random.randn(125, n_features) + np.array(off)
    X_parts.append(data)
    y_parts.extend([name]*125)

X = np.vstack(X_parts)
y = np.array(y_parts)

# Crear DataFrame con nombres de columnas descriptivos
cols = ([f'demo_{i}' for i in range(20)] +
        [f'consumo_{i}' for i in range(20)] +
        [f'digital_{i}' for i in range(20)])

df = pd.DataFrame(X, columns=cols)
df['segmento'] = y
df.to_csv('survey_data.csv', index=False)

print(f'Shape del dataset: {df.shape}')
print('\nDistribución de segmentos:')
print(df['segmento'].value_counts())

## 3. Exploración y limpieza de datos

In [ ]:
# Verificar valores nulos
nulos = df.drop(columns=['segmento']).isnull().sum().sum()
print(f'Valores nulos totales: {nulos}')
print(f'Filas duplicadas: {df.duplicated().sum()}')

# Estadísticas descriptivas resumidas
print('\nRango de valores por tipo de variable:')
for prefix in ['demo', 'consumo', 'digital']:
    subset = df[[c for c in cols if c.startswith(prefix)]]
    print(f'  {prefix}: mean={subset.mean().mean():.2f}, std={subset.std().mean():.2f}')

In [ ]:
# Escalado de variables con StandardScaler (media=0, desv=1)
# Es fundamental antes de PCA y t-SNE para evitar que variables con mayor
# rango dominen los componentes.
X_features = df.drop(columns=['segmento'])
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_features)

print('Escalado completado.')
print(f'Media post-escalado: {X_scaled.mean():.4f}')
print(f'Desv. estándar post-escalado: {X_scaled.std():.4f}')

## 4. Análisis de varianza con PCA (scree plot)
Antes de reducir a 2D, es útil ver cuántos componentes capturan la mayor parte de la varianza.

In [ ]:
pca_full = PCA(random_state=42)
pca_full.fit(X_scaled)
var_acum = np.cumsum(pca_full.explained_variance_ratio_)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(range(1, len(var_acum)+1), var_acum, marker='o', markersize=3, color='#4C72B0')
ax.axhline(0.80, linestyle='--', color='orange', label='80% varianza')
ax.axhline(0.95, linestyle='--', color='red', label='95% varianza')
ax.set_xlabel('Número de componentes', fontsize=12)
ax.set_ylabel('Varianza explicada acumulada', fontsize=12)
ax.set_title('Scree Plot — Varianza explicada acumulada por PCA', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

n80 = np.argmax(var_acum >= 0.80) + 1
n95 = np.argmax(var_acum >= 0.95) + 1
print(f'Componentes para 80% de varianza: {n80}')
print(f'Componentes para 95% de varianza: {n95}')

## 5. Aplicación de PCA (2 componentes)

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
var_exp = pca.explained_variance_ratio_

print(f'Varianza explicada por PC1: {var_exp[0]*100:.1f}%')
print(f'Varianza explicada por PC2: {var_exp[1]*100:.1f}%')
print(f'Total varianza en 2D: {sum(var_exp)*100:.1f}%')

In [ ]:
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

fig, ax = plt.subplots(figsize=(9, 7))
for seg, col, lab in zip(segment_names, colors, labels_pretty):
    mask = y == seg
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], c=col, label=lab,
               alpha=0.75, edgecolors='white', linewidth=0.4, s=60)

ax.set_xlabel(f'PC1 ({var_exp[0]*100:.1f}% varianza)', fontsize=12)
ax.set_ylabel(f'PC2 ({var_exp[1]*100:.1f}% varianza)', fontsize=12)
ax.set_title('PCA — Reducción a 2 Componentes Principales\nVisionData: Segmentos de Clientes', fontsize=13, fontweight='bold')
ax.legend(title='Segmento', fontsize=10, title_fontsize=11, framealpha=0.9)
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('pca_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print('Visualización PCA guardada.')

## 6. Aplicación de t-SNE (2 componentes)

In [ ]:
# t-SNE es estocástico y más lento que PCA.
# perplexity=30 es un valor estándar recomendado para datasets de 100-1000 filas.
tsne = TSNE(n_components=2, perplexity=30, max_iter=1000, random_state=42)
X_tsne = tsne.fit_transform(X_scaled)
print('t-SNE completado.')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
for seg, col, lab in zip(segment_names, colors, labels_pretty):
    mask = y == seg
    ax.scatter(X_tsne[mask, 0], X_tsne[mask, 1], c=col, label=lab,
               alpha=0.75, edgecolors='white', linewidth=0.4, s=60)

ax.set_xlabel('Dimensión t-SNE 1', fontsize=12)
ax.set_ylabel('Dimensión t-SNE 2', fontsize=12)
ax.set_title('t-SNE — Proyección No Lineal 2D\nVisionData: Segmentos de Clientes', fontsize=13, fontweight='bold')
ax.legend(title='Segmento', fontsize=10, title_fontsize=11, framealpha=0.9)
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('tsne_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print('Visualización t-SNE guardada.')

## 7. Comparación de técnicas

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, Xr, title, xlabel, ylabel in [
    (axes[0], X_pca, f'PCA (varianza total: {sum(var_exp)*100:.1f}%)',
     f'PC1 ({var_exp[0]*100:.1f}%)', f'PC2 ({var_exp[1]*100:.1f}%)'),
    (axes[1], X_tsne, 't-SNE (perplexity=30)', 'Dim 1', 'Dim 2')
]:
    for seg, col, lab in zip(segment_names, colors, labels_pretty):
        mask = y == seg
        ax.scatter(Xr[mask, 0], Xr[mask, 1], c=col, label=lab,
                   alpha=0.75, edgecolors='white', linewidth=0.4, s=50)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.legend(title='Segmento', fontsize=9)
    ax.grid(True, linestyle='--', alpha=0.3)

fig.suptitle('Comparación PCA vs t-SNE — VisionData', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Reflexión sobre los métodos

### PCA
- **Ventajas:** Interpretable (los ejes tienen significado), rápido, reproducible, permite proyectar nuevos datos.
- **Limitaciones:** Captura relaciones lineales únicamente. Con 2 componentes solo se retiene ~70% de la varianza total en este dataset.
- **Recomendado para:** Reducción previa a modelos predictivos, explicaciones ejecutivas donde la interpretabilidad importa.

### t-SNE
- **Ventajas:** Excelente para visualizar clusters no lineales y estructuras complejas en 2D/3D.
- **Limitaciones:** No es interpretable (los ejes no tienen unidades), estocástico (resultados varían por semilla), no escalable a millones de registros, no permite proyectar puntos nuevos directamente.
- **Recomendado para:** Exploración visual de agrupamientos naturales.

### ¿Qué haría con volúmenes de datos mucho mayores?
- Usaría **PCA incremental** (`IncrementalPCA`) o `TruncatedSVD` para datos sparse.
- Para visualización de clusters a gran escala, optaría por **UMAP** (más rápido que t-SNE y con mejor preservación de estructura global).
- Aplicaría PCA primero para reducir a ~50 dimensiones y luego t-SNE/UMAP para la proyección 2D final.